# MENG 417 — Homework #4
## Complete, step-by-step MNIST neural-network assignment

**Professor's task:** Use the original **MNIST** raw files to build a **three-fully-connected-layer** neural network, train it on handwritten digits **0–9**, evaluate the untouched test dataset, and prepare **one PDF report with the Python source code appended**.

This is the uploaded **Homework #4**, not Homework #3. This notebook addresses **all four report requirements** and their associated six learning objectives.

### What you will produce

1. **Testing accuracy and a 10 × 10 confusion matrix**, computed from the original 10,000 MNIST test images.
2. **Network structure**: shapes, activations, trainable parameter counts, and a layer diagram.
3. **Loss function**: formula, justification, forward propagation and backpropagation.
4. **Training details**: optimizer, initial learning rate, batch size, epochs, early stopping, Dropout decision, numerical stability, and interpretation.

The final export cell produces a **single PDF containing the four answers, measured results, charts, and the complete experiment source code**. A theory cheat sheet appears at the end of this notebook.

> **Scientific integrity:** Exact test accuracy, confusion-matrix entries and training curves depend on the actual run. This notebook calculates and reports them; it never substitutes an invented score. All displayed mathematics uses `$$ ... $$`.

## Professor's instructions — checked against the uploaded file

| Requirement in Homework #4 | Implementation here |
|---|---|
| Read four **raw** MNIST `.gz` archives | `gzip` + `struct` IDX loader with header and count validation |
| **Do not** call `tf.keras.datasets.mnist.load_data()` | Never called |
| 28 × 28 grayscale inputs, 10 digits | Each image becomes 784 numerical inputs |
| Exactly **three Dense layers** | Dense(128, ReLU) → Dense(64, ReLU) → Dense(10, Softmax) |
| Test accuracy and confusion matrix | Computed on original 10,000 examples after training |
| Network structure | Mathematical diagram, shape checks, exact parameter count |
| Loss function | Sparse categorical cross-entropy with explanation |
| Optimizer, initial learning rate, training settings, Dropout | Documented and reported from execution |
| **One PDF with code appended** | Generated from the actual results with a code appendix |

**Design choices** (not fixed by the handout): 128 and 64 hidden units, ReLU, Adam, learning rate 0.001, batch size 128, up to 15 epochs, 10,000-image internal validation partition, and **no Dropout**. The professor permits choosing these settings, and specifically states that Dropout is not required.

# Part 0 — Background and workflow

MNIST contains **60,000 development/training images** and **10,000 separate test images**. We split the original 60,000 into 50,000 for weight fitting and 10,000 for validation. This is a **modeling choice**, not an instruction explicitly prescribing a validation split.

$$
\underbrace{60{,}000}_{\text{official training split}}
\longrightarrow
\underbrace{50{,}000}_{\text{fit weights}} +
\underbrace{10{,}000}_{\text{validate and select epoch}}
$$

$$
\underbrace{10{,}000}_{\text{official test split}}
\longrightarrow
\text{final evaluation only}
$$

**Analogy:** Training is practice; validation is a mock exam used to decide when to stop studying; the official test is the final exam, not used to tune the model.

Before you begin in Colab, select **Runtime → Change runtime type → Python 3** (GPU optional). Then choose **Runtime → Run all**. The first download may require internet access; alternatively set `DATA_MODE = "upload"` and provide the original four `.gz` files.

In [ ]:
# Part 0: reproducible environment and paths
import gzip
import os
import random
import struct
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 417
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

OUT_DIR = Path('MENG417_HW4_Results')
OUT_DIR.mkdir(exist_ok=True)
DATA_DIR = Path('mnist_raw')
DATA_DIR.mkdir(exist_ok=True)

# 'auto': fetch original raw archives from the publicly hosted MNIST IDX mirror
# 'upload': upload all four original archives in Google Colab
DATA_MODE = 'auto'

print('TensorFlow:', tf.__version__)
print('Dataset mode:', DATA_MODE)
print('Output folder:', OUT_DIR.resolve())

# Part 1 — Acquire the four original MNIST archives

The professor specifically prohibits loading the dataset through TensorFlow's built-in MNIST loader. We instead read the four original **IDX gzip** files:

`train-images-idx3-ubyte.gz`, `train-labels-idx1-ubyte.gz`, `t10k-images-idx3-ubyte.gz`, and `t10k-labels-idx1-ubyte.gz`.

The public archive mirror is used **only to download those raw files**. No framework dataset-loading function is involved. If the download fails, run this cell with `DATA_MODE='upload'` and select the four official files.

In [ ]:
RAW_FILES = {
    'train_images': 'train-images-idx3-ubyte.gz',
    'train_labels': 'train-labels-idx1-ubyte.gz',
    'test_images':  't10k-images-idx3-ubyte.gz',
    'test_labels':  't10k-labels-idx1-ubyte.gz',
}
RAW_BASE_URL = 'https://storage.googleapis.com/cvdf-datasets/mnist/'

if DATA_MODE == 'upload':
    from google.colab import files
    print('Upload all four original MNIST .gz files in one selection.')
    uploaded = files.upload()
    for filename, contents in uploaded.items():
        if filename in RAW_FILES.values():
            (DATA_DIR / filename).write_bytes(contents)
elif DATA_MODE == 'auto':
    for filename in RAW_FILES.values():
        dest = DATA_DIR / filename
        if not dest.exists() or dest.stat().st_size == 0:
            try:
                print('Downloading:', filename)
                urlretrieve(RAW_BASE_URL + filename, str(dest))
            except Exception as exc:
                if dest.exists():
                    dest.unlink()
                raise RuntimeError(
                    f'Cannot download {filename}. Set DATA_MODE="upload" and '
                    'upload the original four MNIST archives.'
                ) from exc
else:
    raise ValueError('DATA_MODE must be auto or upload')

for filename in RAW_FILES.values():
    dest = DATA_DIR / filename
    if not dest.is_file() or dest.stat().st_size == 0:
        raise FileNotFoundError(f'Missing required MNIST archive: {dest}')
    print(f'{filename:36s}  {dest.stat().st_size:,} bytes')

## Mini-step 1.1 — Read IDX headers correctly

An image archive begins with a **16-byte header**:

$$
(\text{magic number},\ \text{number of images},\ 28,\ 28)
$$

The standard MNIST image magic number is **2051**. The label archive begins with an **8-byte header**; its magic number is **2049**.

For each image, the 28 × 28 pixels are unsigned bytes (0–255). We reshape the array to

$$
X_{\mathrm{raw}}\in\mathbb{R}^{N\times784}.
$$

This loader checks the headers and payload length so errors in the input archives cannot silently change the model.

In [ ]:
def read_idx_images(path):
    with gzip.open(path, 'rb') as f:
        header = f.read(16)
        if len(header) != 16:
            raise ValueError(f'Incomplete MNIST image header: {path}')
        magic, count, rows, cols = struct.unpack('>IIII', header)
        if magic != 2051 or (rows, cols) != (28, 28):
            raise ValueError(f'Invalid MNIST image header: {(magic, count, rows, cols)}')
        payload = f.read()
    result = np.frombuffer(payload, dtype=np.uint8)
    if result.size != count * rows * cols:
        raise ValueError(f'Pixel payload length mismatch in {path}')
    return result.reshape(count, rows * cols).copy()


def read_idx_labels(path):
    with gzip.open(path, 'rb') as f:
        header = f.read(8)
        if len(header) != 8:
            raise ValueError(f'Incomplete MNIST label header: {path}')
        magic, count = struct.unpack('>II', header)
        if magic != 2049:
            raise ValueError(f'Invalid MNIST label magic number: {magic}')
        payload = f.read()
    labels = np.frombuffer(payload, dtype=np.uint8)
    if labels.size != count:
        raise ValueError(f'Label payload length mismatch in {path}')
    return labels.copy()


def loadMNIST(data_dir=DATA_DIR):
    train_images = read_idx_images(data_dir / RAW_FILES['train_images'])
    train_labels = read_idx_labels(data_dir / RAW_FILES['train_labels'])
    test_images = read_idx_images(data_dir / RAW_FILES['test_images'])
    test_labels = read_idx_labels(data_dir / RAW_FILES['test_labels'])
    if len(train_images) != len(train_labels) or len(test_images) != len(test_labels):
        raise ValueError('Image and label counts do not match')
    return train_images, train_labels, test_images, test_labels

train_images_raw, train_labels_raw, test_images_raw, test_labels_raw = loadMNIST()

print('Train images:', train_images_raw.shape)
print('Train labels:', train_labels_raw.shape)
print('Test images: ', test_images_raw.shape)
print('Test labels: ', test_labels_raw.shape)

assert train_images_raw.shape == (60000, 784)
assert train_labels_raw.shape == (60000,)
assert test_images_raw.shape == (10000, 784)
assert test_labels_raw.shape == (10000,)
assert set(np.unique(train_labels_raw)) == set(range(10))
assert set(np.unique(test_labels_raw)) == set(range(10))
print('PASS: original MNIST archive shapes and label range verified.')

## Mini-step 1.2 — Visually inspect the digits

Every row contains 784 pixel values. To display a digit as a two-dimensional image, reverse the flattening:

$$
\mathbf{x}\in\mathbb{R}^{784}
\quad\longrightarrow\quad
X_{\mathrm{image}}\in\mathbb{R}^{28\times28}.
$$

The labels range from **0 to 9**. Unlike the earlier binary perceptron lab, **we keep all ten classes** in Homework #4.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(train_images_raw[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Label: {int(train_labels_raw[i])}')
    ax.axis('off')
fig.suptitle('First ten images in the original MNIST training data', fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / 'mnist_examples.png', dpi=170, bbox_inches='tight')
plt.show()

print('Training class counts:', np.bincount(train_labels_raw, minlength=10))
print('Test class counts:    ', np.bincount(test_labels_raw, minlength=10))

# Part 2 — Preprocess the images and make a clean validation split

### Mini-step 2.1 — Normalize pixels

Raw pixel values lie between 0 and 255. Divide by 255 to place values between 0 and 1:

$$
\boxed{x_{\mathrm{scaled}}=\frac{x_{\mathrm{raw}}}{255}}
$$

This makes optimization more numerically manageable. We use `float32` because it is supported efficiently by TensorFlow.

### Mini-step 2.2 — Separate weight training and validation

Use a reproducible **stratified** split of the official training data:

- **50,000** images to fit the weights;
- **10,000** images to monitor validation accuracy/loss and choose the stopping epoch;
- keep the official **10,000** test images untouched until final evaluation.

Stratification preserves the approximate relative representation of digits 0–9 in each development partition.

In [ ]:
all_indices = np.arange(train_labels_raw.shape[0])
fit_idx, val_idx = train_test_split(
    all_indices,
    test_size=10000,
    random_state=SEED,
    stratify=train_labels_raw,
)

x_fit = train_images_raw[fit_idx].astype(np.float32) / 255.0
y_fit = train_labels_raw[fit_idx].astype(np.int64)
x_val = train_images_raw[val_idx].astype(np.float32) / 255.0
y_val = train_labels_raw[val_idx].astype(np.int64)
x_test = test_images_raw.astype(np.float32) / 255.0
y_test = test_labels_raw.astype(np.int64)

for name, x, y in [
    ('Training', x_fit, y_fit),
    ('Validation', x_val, y_val),
    ('Test', x_test, y_test),
]:
    print(f'{name:11s}: images={x.shape}, labels={y.shape}, '
          f'range=[{x.min():.3f}, {x.max():.3f}]')
    assert x.ndim == 2 and x.shape[1] == 784
    assert x.dtype == np.float32
    assert y.min() >= 0 and y.max() <= 9

assert (len(x_fit), len(x_val), len(x_test)) == (50000, 10000, 10000)
assert set(fit_idx).isdisjoint(set(val_idx))
print('PASS: train/validation/test partitions and normalization.')

# Requirement 2 — Construct and explain the network structure

The professor requires **exactly three fully connected layers**, with ten output neurons. We choose:

$$
\boxed{784\ \longrightarrow\ 128\ \longrightarrow\ 64\ \longrightarrow\ 10}
$$

$$
\underbrace{\mathbf{x}\in\mathbb{R}^{784}}_{\text{28 × 28 grayscale pixels}}
\;\longrightarrow\;
\underbrace{\mathrm{Dense}(128,\mathrm{ReLU})}_{\text{first fully connected layer}}
\;\longrightarrow\;
\underbrace{\mathrm{Dense}(64,\mathrm{ReLU})}_{\text{second fully connected layer}}
\;\longrightarrow\;
\underbrace{\mathrm{Dense}(10,\mathrm{Softmax})}_{\text{class probabilities}}.
$$

**Why these choices?** ReLU introduces nonlinearity so two hidden layers can learn more complex visual features than one linear classifier. Softmax produces ten nonnegative numbers that sum to 1. Ten neurons are required because the digits are 0 through 9.

### Mini-step — Calculate the trainable parameters by hand

A Dense layer with $d$ inputs and $h$ neurons has $dh$ weights plus $h$ biases:

$$
N_{\mathrm{parameters}}=(d+1)h.
$$

| Fully connected layer | Calculation | Parameters |
|---|---:|---:|
| Input 784 → Hidden 128 | $(784+1)\times128$ | **100,480** |
| Hidden 128 → Hidden 64 | $(128+1)\times64$ | **8,256** |
| Hidden 64 → Output 10 | $(64+1)\times10$ | **650** |
| **Total** | | **109,386** |

We do not include a Dropout, convolutional, pooling, or batch-normalization layer; the model contains exactly three trainable Dense layers.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,), name='input_784'),
    tf.keras.layers.Dense(
        128, activation='relu', kernel_initializer='he_normal', name='dense_128'
    ),
    tf.keras.layers.Dense(
        64, activation='relu', kernel_initializer='he_normal', name='dense_64'
    ),
    tf.keras.layers.Dense(
        10, activation='softmax', name='class_10'
    ),
], name='MNIST_3_Dense_Network')

model.summary()
actual_dense_layers = [
    layer for layer in model.layers if isinstance(layer, tf.keras.layers.Dense)
]
assert len(actual_dense_layers) == 3
assert [layer.units for layer in actual_dense_layers] == [128, 64, 10]
assert model.count_params() == 109386
assert model.output_shape[-1] == 10
assert not any(isinstance(layer, tf.keras.layers.Dropout) for layer in model.layers)

print('PASS: three Dense layers; 109,386 trainable parameters; no Dropout.')

In [ ]:
# Layer-structure table for the professor's report
layer_details = pd.DataFrame([
    {
        'Layer': layer.name,
        'Neurons': layer.units,
        'Activation': layer.activation.__name__,
        'Trainable parameters': layer.count_params(),
    }
    for layer in actual_dense_layers
])
print(layer_details.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 2.7))
ax.axis('off')
architecture = [
    ('Input', '784 pixels'),
    ('Dense 1', '128 · ReLU'),
    ('Dense 2', '64 · ReLU'),
    ('Dense 3', '10 · Softmax'),
]
for i, (title, detail) in enumerate(architecture):
    x = 0.115 + i * 0.25
    ax.text(x, 0.6, title, ha='center', fontsize=12, weight='bold',
            bbox=dict(boxstyle='round,pad=.7', fc='#eef3f9', ec='#38577a'))
    ax.text(x, 0.26, detail, ha='center', fontsize=11)
    if i < 3:
        ax.annotate('', xy=(x+0.21, .59), xytext=(x+0.10, .59),
                    arrowprops=dict(arrowstyle='->', lw=1.5))
fig.suptitle('Three fully connected layers (input not counted as a Dense layer)',
             fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR / 'architecture.png', dpi=180, bbox_inches='tight')
plt.show()

# Requirement 3 — Explain forward propagation and the loss function

## Mini-step 3.1 — Forward propagation

For an input column vector $\mathbf{x}\in\mathbb{R}^{784}$, the network performs:

$$
\mathbf z_1=W_1^T\mathbf x+\mathbf b_1,
\qquad
\mathbf a_1=\mathrm{ReLU}(\mathbf z_1),
$$

$$
\mathbf z_2=W_2^T\mathbf a_1+\mathbf b_2,
\qquad
\mathbf a_2=\mathrm{ReLU}(\mathbf z_2),
$$

$$
\mathbf z_3=W_3^T\mathbf a_2+\mathbf b_3,
\qquad
p_k=\frac{e^{z_{3,k}}}{\sum_{j=0}^{9}e^{z_{3,j}}}.
$$

Here:

$$
W_1\in\mathbb R^{784\times128},\quad
W_2\in\mathbb R^{128\times64},\quad
W_3\in\mathbb R^{64\times10}.
$$

The predicted digit is the index with the largest probability:

$$
\boxed{\hat y=\underset{k\in\{0,\ldots,9\}}{\arg\max}\;p_k.}
$$

**Everyday analogy:** The first layer examines combinations of pixels, the second combines these signals into more informative patterns, and the last layer votes between ten possible digits.

## Mini-step 3.2 — Choose the loss function

Because the labels are integer categories $0,\ldots,9$, use **sparse categorical cross-entropy**:

$$
\boxed{\ell(\mathbf p,y)=-\log(p_y).}
$$

For a batch of $B$ examples:

$$
\boxed{L=\frac{1}{B}\sum_{i=1}^{B}-\log p_{i,y_i}.}
$$

This is preferable to mean squared error for this ten-class probability classification problem: it directly penalizes low probability assigned to the correct class. We use the sparse version because integer labels do **not** require one-hot encoding.

**Numerical example:** If the true digit is 7 and the model gives $p_7=0.8$:

$$
\ell=-\ln(0.8)\approx0.2231.
$$

If it gives $p_7=0.01$, the loss becomes $4.6052$, which is a much stronger penalty.

## Mini-step 3.3 — Backpropagation using the chain rule

The model learns by computing derivatives of the loss with respect to all weights. For softmax followed by categorical cross-entropy, the derivative with respect to each output logit has an elegant form:

$$
\boxed{\delta_{3,k}=p_k-\mathbf1\{y=k\}.}
$$

This is the prediction probability minus the true one-hot target for each output neuron.

For one training example, the gradients for the last layer are:

$$
\frac{\partial\ell}{\partial W_3}
=\mathbf a_2\boldsymbol\delta_3^T,
\qquad
\frac{\partial\ell}{\partial\mathbf b_3}
=\boldsymbol\delta_3.
$$

Propagate the error backwards through the second hidden layer:

$$
\boldsymbol\delta_2=
\left(W_3\boldsymbol\delta_3\right)
\odot \mathrm{ReLU}'(\mathbf z_2).
$$

Then:

$$
\frac{\partial\ell}{\partial W_2}=
\mathbf a_1\boldsymbol\delta_2^T.
$$

Likewise, the first hidden-layer error is

$$
\boldsymbol\delta_1=
\left(W_2\boldsymbol\delta_2\right)
\odot \mathrm{ReLU}'(\mathbf z_1),
$$

and the first-layer weight gradient is

$$
\frac{\partial\ell}{\partial W_1}
=\mathbf x\boldsymbol\delta_1^T.
$$

Here $\odot$ means elementwise multiplication, and $\mathrm{ReLU}'(z)=1$ for $z>0$ and $0$ for $z<0$ (with a chosen subgradient at zero).

**Computational graph:**

$$
\mathbf x\ \rightarrow\ \mathbf z_1\ \rightarrow\ \mathbf a_1\
\rightarrow\ \mathbf z_2\ \rightarrow\ \mathbf a_2\
\rightarrow\ \mathbf z_3\ \rightarrow\ \mathbf p\ \rightarrow\ \ell.
$$

Backpropagation follows those links **in reverse**, applying the chain rule. TensorFlow computes these gradients automatically when we call `model.fit()`.

# Requirement 4 — Training design and optimizer settings

We choose the following settings before looking at test results:

| Setting | Selected value | Rationale |
|---|---|---|
| Optimizer | **Adam** | Adaptive learning rates, useful baseline for MLPs |
| **Initial learning rate** | **0.001** | Standard moderate starting point for Adam |
| Batch size | **128** | Efficient mini-batch updates |
| Maximum epochs | **15** | Allows convergence without unlimited fitting |
| Early stopping | Validation loss, patience **3**, restore best | Limits unnecessary additional training |
| Activation | ReLU hidden, Softmax output | Nonlinear features and multiclass probabilities |
| Weight initialization | **He normal** for ReLU layers | Helps maintain reasonable activation scales |
| Dropout | **No** | Optional in the professor's task; preserves exactly three Dense layers |
| Data scaling | Pixel / 255 | Stable, consistent input magnitude |

The Adam optimizer combines running estimates of the gradient and squared gradient. Conceptually,

$$
\mathbf w_{t+1}=\mathbf w_t-
\eta\frac{\widehat{\mathbf m}_t}{\sqrt{\widehat{\mathbf v}_t}+\epsilon}.
$$

TensorFlow/Keras handles backpropagation, gradient aggregation and Adam updates.

**Early stopping** monitors **validation loss**, not test accuracy. The untouched test set will be evaluated only after `model.fit` finishes and the best validation checkpoint is restored.

In [ ]:
INITIAL_LEARNING_RATE = 0.001
BATCH_SIZE = 128
MAX_EPOCHS = 15
EARLY_STOP_PATIENCE = 3
DROPOUT_USED = False

optimizer = tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=EARLY_STOP_PATIENCE,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.TerminateOnNaN(),
]

print('Optimizer: Adam')
print('Initial learning rate:', INITIAL_LEARNING_RATE)
print('Loss:', type(loss_fn).__name__)
print('Epoch limit:', MAX_EPOCHS, '; batch size:', BATCH_SIZE)
print('EarlyStopping patience:', EARLY_STOP_PATIENCE)
print('Dropout used:', DROPOUT_USED)

# Mini-step 4.1 — Train the model

At every mini-batch:

1. Send the 784 input pixels through the **three Dense layers**.
2. Compute probabilities for digits 0–9.
3. Calculate sparse categorical cross-entropy against the true labels.
4. Differentiate the loss through all layers using **backpropagation**.
5. Use Adam to update weights and biases.
6. Monitor the validation loss at the end of the epoch.

**Do not report the test accuracy yet.** It is only computed after the training and validation decisions have been completed.

In [ ]:
history = model.fit(
    x_fit, y_fit,
    validation_data=(x_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    shuffle=True,
    verbose=2,
)

observed_epochs = len(history.history['loss'])
best_epoch = int(np.argmin(history.history['val_loss']) + 1)
print('Epochs actually run:', observed_epochs)
print('Best validation-loss epoch:', best_epoch)

## Mini-step 4.2 — Plot training and validation curves

These graphs are necessary to explain model quality and identify signs of **underfitting** or **overfitting**.

- **Underfitting:** both training and validation results remain poor; the model may be too simple or undertrained.
- **Overfitting:** training improves, but validation loss rises or validation accuracy stagnates; generalization becomes worse.
- **Good fit:** both curves improve and remain reasonably close.

A single curve or a small gap does not by itself prove that a model will generalize; the untouched test set is still required for an independent estimate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
epochs = np.arange(1, observed_epochs + 1)

axes[0].plot(epochs, history.history['loss'], marker='o', label='Training')
axes[0].plot(epochs, history.history['val_loss'], marker='o', label='Validation')
axes[0].set_title('Sparse categorical cross-entropy')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history.history['accuracy'], marker='o', label='Training')
axes[1].plot(epochs, history.history['val_accuracy'], marker='o', label='Validation')
axes[1].set_title('Classification accuracy')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (fraction)')
axes[1].legend(); axes[1].grid(alpha=0.3)

fig.suptitle('MNIST three-Dense-layer training history')
fig.tight_layout()
fig.savefig(OUT_DIR / 'learning_curves.png', dpi=180, bbox_inches='tight')
plt.show()

print('Final logged training accuracy: ', f"{history.history['accuracy'][-1]:.2%}")
print('Final logged validation accuracy:', f"{history.history['val_accuracy'][-1]:.2%}")
print('Best validation loss:', f"{min(history.history['val_loss']):.6f}")

# Requirement 1 — Evaluate the final model on the 10,000-image test set

This is the **first time** we evaluate the untouched official test set.

### Test accuracy

$$
\boxed{\mathrm{Accuracy}_{\mathrm{test}}=
\frac{\#\{\hat y_i=y_i\}}{10{,}000}\times100\%.}
$$

### Confusion matrix

Let $C_{ij}$ denote the number of images whose **true digit is $i$** and whose **predicted digit is $j$**:

$$
\boxed{C_{ij}=\#\{n:y_n=i,\ \hat y_n=j\},\quad i,j\in\{0,\ldots,9\}.}
$$

The confusion matrix must contain ten rows and ten columns, and **all 100 entries together must sum to 10,000**. Values on the diagonal are correct predictions; off-diagonal values identify misclassifications.

In [ ]:
# Evaluate once after training/model selection.
test_loss, test_accuracy_fraction = model.evaluate(
    x_test, y_test, batch_size=512, verbose=0
)

probabilities = model.predict(x_test, batch_size=512, verbose=0)
y_pred = np.argmax(probabilities, axis=1).astype(np.int64)
cm = confusion_matrix(y_test, y_pred, labels=np.arange(10))

number_correct = int(np.count_nonzero(y_pred == y_test))
test_accuracy_percent = 100.0 * number_correct / len(y_test)

assert cm.shape == (10, 10)
assert int(cm.sum()) == len(y_test) == 10000
assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-4)
assert abs(test_accuracy_percent/100 - test_accuracy_fraction) < 2e-3

print('OFFICIAL MNIST TEST RESULTS')
print('Test cross-entropy:', f'{test_loss:.6f}')
print('Correct predictions:', f'{number_correct:,} / 10,000')
print('TEST ACCURACY:', f'{test_accuracy_percent:.2f}%')
print('\nConfusion matrix (rows = actual; columns = predicted):')
print(pd.DataFrame(cm, index=np.arange(10), columns=np.arange(10)).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7.5))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=ax, fraction=.045)
ax.set(
    xticks=np.arange(10), yticks=np.arange(10),
    xlabel='Predicted digit', ylabel='True digit',
    title='MNIST 10 × 10 test confusion matrix',
)
threshold = cm.max() / 2
for row in range(10):
    for col in range(10):
        ax.text(col, row, str(cm[row, col]), ha='center', va='center',
                fontsize=9, color='white' if cm[row, col] > threshold else 'black')
fig.tight_layout()
fig.savefig(OUT_DIR / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

### Mini-step — Interpret the confusion matrix

For digit $k$, the class recall is

$$
\boxed{\mathrm{Recall}_k=\frac{C_{kk}}{\sum_{j=0}^{9}C_{kj}}.}
$$

This answers: **Of the test images that truly show digit $k$, what fraction does the network classify correctly?** The next cell generates class-by-class precision, recall and F1 values.

In [ ]:
report_dictionary = classification_report(
    y_test, y_pred, labels=list(range(10)),
    output_dict=True, zero_division=0,
)
per_digit = pd.DataFrame([
    {
        'Digit': digit,
        'Test count': int(cm[digit].sum()),
        'Correct': int(cm[digit, digit]),
        'Recall': float(report_dictionary[str(digit)]['recall']),
        'Precision': float(report_dictionary[str(digit)]['precision']),
        'F1': float(report_dictionary[str(digit)]['f1-score']),
    }
    for digit in range(10)
])
print(per_digit.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
per_digit.to_csv(OUT_DIR / 'per_digit_metrics.csv', index=False)
np.savetxt(OUT_DIR / 'confusion_matrix.csv', cm, fmt='%d', delimiter=',')

In [ ]:
wrong = np.flatnonzero(y_pred != y_test)
fig, axes = plt.subplots(2, 5, figsize=(10.5, 5))
for ax, idx in zip(axes.ravel(), wrong[:10]):
    ax.imshow(x_test[idx].reshape(28,28), cmap='gray')
    ax.set_title(f'True {y_test[idx]}  /  Pred {y_pred[idx]}', fontsize=10)
    ax.axis('off')
for ax in axes.ravel()[len(wrong[:10]):]:
    ax.axis('off')
fig.suptitle('Examples of incorrect test predictions (when present)')
fig.tight_layout()
fig.savefig(OUT_DIR / 'mistakes.png', dpi=170, bbox_inches='tight')
plt.show()
print('Number of test mistakes:', len(wrong))

# Requirements 3 and 4 — Explain model choice, overfitting, dropout and numerical stability

### Model selection, underfitting and overfitting

Our three-layer network has **109,386 trainable parameters**. More model capacity can fit more complex patterns but may also increase overfitting risk. We made the architecture choice **before** looking at the test set, and then used **validation loss** and early stopping to decide which training epoch to retain.

A high training accuracy with significantly lower validation accuracy can suggest overfitting; poor training and validation accuracy together may suggest underfitting. The actual conclusions should be based on the curves generated above.

### Dropout decision

**No Dropout was used.** The professor writes that additional types of layers, including Dropout, are not required. A dropout layer would randomly suppress a specified fraction of hidden activations **during training**, discouraging co-adaptation of neurons; at inference time dropout is disabled and appropriate scaling is handled by the framework. Using Dropout is not inherently guaranteed to improve accuracy. We instead monitor validation loss and restore the best epoch.

### Numerical stability

- **Pixel scaling:** values are reduced from 0–255 to 0–1.
- **He initialization:** helps keep activations at a manageable scale in ReLU layers.
- **Cross-entropy and Softmax:** handled through TensorFlow/Keras operations, avoiding naive manual products of tiny probabilities.
- **Adam** uses an $\epsilon$ term to avoid division by zero in adaptive updates.
- **TerminateOnNaN** stops training if the loss becomes non-finite.

### Important distinction

**Softmax probabilities** are model estimates, not automatically perfectly calibrated probabilities. Test accuracy evaluates classification correctness, while validation curves help us interpret training behavior.

In [ ]:
# Dynamic, evidence-based comments; do not hard-code a model quality claim.
last_fit_acc = float(history.history['accuracy'][-1])
last_val_acc = float(history.history['val_accuracy'][-1])
gap = last_fit_acc - last_val_acc

print('TRAINING DIAGNOSTICS')
print(f'Final logged training accuracy: {last_fit_acc:.2%}')
print(f'Final logged validation accuracy: {last_val_acc:.2%}')
print(f'Train minus validation accuracy: {gap:+.2%}')
print(f'Best validation-loss epoch: {best_epoch} of {observed_epochs} run')
if gap > 0.03:
    print('Interpretation: a >3 percentage-point gap may indicate overfitting.')
elif last_fit_acc < 0.85 and last_val_acc < 0.85:
    print('Interpretation: both accuracies are low; inspect underfitting or training setup.')
else:
    print('Interpretation: no strong accuracy-gap warning by this simple heuristic.')
print('This heuristic is not a formal statistical test of generalization.')

# Four professor-ready answers (automatically populated)

The next cell assembles the **actual measured** results into four short report answers. These statements are also included in the generated PDF; no need to manually transcribe a training accuracy or guess a confusion-matrix value.

In [ ]:
report_summary = {
    'test_accuracy_percent': float(test_accuracy_percent),
    'test_loss': float(test_loss),
    'correct': int(number_correct),
    'test_total': int(len(y_test)),
    'confusion_matrix': cm.copy(),
    'network': '784 inputs -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(10, Softmax)',
    'parameter_count': int(model.count_params()),
    'loss_name': 'Sparse categorical cross-entropy',
    'optimizer': 'Adam',
    'initial_learning_rate': INITIAL_LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'epochs_run': observed_epochs,
    'epoch_limit': MAX_EPOCHS,
    'best_validation_epoch': best_epoch,
    'validation_patience': EARLY_STOP_PATIENCE,
    'dropout_used': False,
    'final_training_accuracy': last_fit_acc,
    'final_validation_accuracy': last_val_acc,
}

print('REQUIREMENT 1 — TEST ACCURACY AND CONFUSION MATRIX')
print(f"Accuracy: {report_summary['test_accuracy_percent']:.2f}% "
      f"({report_summary['correct']:,}/{report_summary['test_total']:,})")
print(pd.DataFrame(cm).to_string(index=False, header=False))

print('\nREQUIREMENT 2 — NETWORK STRUCTURE')
print(report_summary['network'])
print('Trainable parameters:', report_summary['parameter_count'])

print('\nREQUIREMENT 3 — LOSS')
print('Sparse categorical cross-entropy (integer labels 0–9)')

print('\nREQUIREMENT 4 — TRAINING / DROPOUT')
print('Adam, initial lr=', INITIAL_LEARNING_RATE, ', batch=', BATCH_SIZE,
      ', maximum epochs=', MAX_EPOCHS, ', observed epochs=', observed_epochs)
print('Early stopping: validation loss, patience 3; best epoch=', best_epoch)
print('Dropout: Not used; professor says optional.')

# Final numerical and instruction-compliance audit

This validation is deliberately independent of the figures: the predictions, probability vectors, confusion matrix and parameter count must all agree numerically.

In [ ]:
assert train_images_raw.shape == (60000, 784)
assert test_images_raw.shape == (10000, 784)
assert len(actual_dense_layers) == 3
assert [layer.units for layer in actual_dense_layers] == [128, 64, 10]
assert model.count_params() == 109386
assert np.all(np.isfinite(probabilities))
assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)
assert cm.shape == (10, 10) and cm.sum() == 10000
assert np.array_equal(cm.sum(axis=1), np.bincount(y_test, minlength=10))
assert cm.trace() == number_correct
assert abs(cm.trace()/cm.sum() - test_accuracy_percent/100) < 1e-12
assert not DROPOUT_USED

print('ALL AVAILABLE MATHEMATICAL AND INSTRUCTION-COMPLIANCE CHECKS PASSED.')
print('These checks verify internal consistency, not a guaranteed professor grade.')

# Submission export — One PDF with results and Python source appended

The professor explicitly asks to submit **one PDF** with the source code at the end. The cell below assembles one PDF from the **measurements you have just generated**:

- Cover and a four-requirement report;
- ten-digit test accuracy and confusion matrix;
- the architecture and exact trainable parameter counts;
- cross-entropy, optimizer, initial learning rate, hyperparameters and Dropout discussion;
- learning curves, error examples and analysis;
- a complete appendix containing the core experiment Python source code.

This cell also writes the `.py` companion file. **Only the generated combined PDF is required by the professor's submission instruction.**

In [ ]:
# Use the real measured metrics and generated plots, never placeholder figures.
import sys
import subprocess
try:
    import reportlab
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'reportlab'], check=True)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.lib.enums import TA_CENTER
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                TableStyle, Image, Preformatted, CondPageBreak,
                                KeepTogether)

# Contains the exact code of the notebook's core experiment cells.
EXPERIMENT_SOURCE = '# MENG 417 Homework #4 — Complete MNIST experiment source code\n# Generated from the Google Colab experiment cells.\n# In Colab, run this script with Python after TensorFlow and sklearn are installed.\n\n\n\n# ==========================================================================\n# 0 — Imports, random seed, configuration\n# ==========================================================================\n\n# Part 0: reproducible environment and paths\nimport gzip\nimport os\nimport random\nimport struct\nfrom pathlib import Path\nfrom urllib.request import urlretrieve\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib.pyplot as plt\nimport tensorflow as tf\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import confusion_matrix, classification_report\n\nSEED = 417\nrandom.seed(SEED)\nnp.random.seed(SEED)\ntf.keras.utils.set_random_seed(SEED)\n\nOUT_DIR = Path(\'MENG417_HW4_Results\')\nOUT_DIR.mkdir(exist_ok=True)\nDATA_DIR = Path(\'mnist_raw\')\nDATA_DIR.mkdir(exist_ok=True)\n\n# \'auto\': fetch original raw archives from the publicly hosted MNIST IDX mirror\n# \'upload\': upload all four original archives in Google Colab\nDATA_MODE = \'auto\'\n\nprint(\'TensorFlow:\', tf.__version__)\nprint(\'Dataset mode:\', DATA_MODE)\nprint(\'Output folder:\', OUT_DIR.resolve())\n\n\n# ==========================================================================\n# 1 — Acquire raw MNIST archives\n# ==========================================================================\n\nRAW_FILES = {\n    \'train_images\': \'train-images-idx3-ubyte.gz\',\n    \'train_labels\': \'train-labels-idx1-ubyte.gz\',\n    \'test_images\':  \'t10k-images-idx3-ubyte.gz\',\n    \'test_labels\':  \'t10k-labels-idx1-ubyte.gz\',\n}\nRAW_BASE_URL = \'https://storage.googleapis.com/cvdf-datasets/mnist/\'\n\nif DATA_MODE == \'upload\':\n    from google.colab import files\n    print(\'Upload all four original MNIST .gz files in one selection.\')\n    uploaded = files.upload()\n    for filename, contents in uploaded.items():\n        if filename in RAW_FILES.values():\n            (DATA_DIR / filename).write_bytes(contents)\nelif DATA_MODE == \'auto\':\n    for filename in RAW_FILES.values():\n        dest = DATA_DIR / filename\n        if not dest.exists() or dest.stat().st_size == 0:\n            try:\n                print(\'Downloading:\', filename)\n                urlretrieve(RAW_BASE_URL + filename, str(dest))\n            except Exception as exc:\n                if dest.exists():\n                    dest.unlink()\n                raise RuntimeError(\n                    f\'Cannot download {filename}. Set DATA_MODE="upload" and \'\n                    \'upload the original four MNIST archives.\'\n                ) from exc\nelse:\n    raise ValueError(\'DATA_MODE must be auto or upload\')\n\nfor filename in RAW_FILES.values():\n    dest = DATA_DIR / filename\n    if not dest.is_file() or dest.stat().st_size == 0:\n        raise FileNotFoundError(f\'Missing required MNIST archive: {dest}\')\n    print(f\'{filename:36s}  {dest.stat().st_size:,} bytes\')\n\n\n# ==========================================================================\n# 2 — Parse and validate IDX archives\n# ==========================================================================\n\ndef read_idx_images(path):\n    with gzip.open(path, \'rb\') as f:\n        header = f.read(16)\n        if len(header) != 16:\n            raise ValueError(f\'Incomplete MNIST image header: {path}\')\n        magic, count, rows, cols = struct.unpack(\'>IIII\', header)\n        if magic != 2051 or (rows, cols) != (28, 28):\n            raise ValueError(f\'Invalid MNIST image header: {(magic, count, rows, cols)}\')\n        payload = f.read()\n    result = np.frombuffer(payload, dtype=np.uint8)\n    if result.size != count * rows * cols:\n        raise ValueError(f\'Pixel payload length mismatch in {path}\')\n    return result.reshape(count, rows * cols).copy()\n\n\ndef read_idx_labels(path):\n    with gzip.open(path, \'rb\') as f:\n        header = f.read(8)\n        if len(header) != 8:\n            raise ValueError(f\'Incomplete MNIST label header: {path}\')\n        magic, count = struct.unpack(\'>II\', header)\n        if magic != 2049:\n            raise ValueError(f\'Invalid MNIST label magic number: {magic}\')\n        payload = f.read()\n    labels = np.frombuffer(payload, dtype=np.uint8)\n    if labels.size != count:\n        raise ValueError(f\'Label payload length mismatch in {path}\')\n    return labels.copy()\n\n\ndef loadMNIST(data_dir=DATA_DIR):\n    train_images = read_idx_images(data_dir / RAW_FILES[\'train_images\'])\n    train_labels = read_idx_labels(data_dir / RAW_FILES[\'train_labels\'])\n    test_images = read_idx_images(data_dir / RAW_FILES[\'test_images\'])\n    test_labels = read_idx_labels(data_dir / RAW_FILES[\'test_labels\'])\n    if len(train_images) != len(train_labels) or len(test_images) != len(test_labels):\n        raise ValueError(\'Image and label counts do not match\')\n    return train_images, train_labels, test_images, test_labels\n\ntrain_images_raw, train_labels_raw, test_images_raw, test_labels_raw = loadMNIST()\n\nprint(\'Train images:\', train_images_raw.shape)\nprint(\'Train labels:\', train_labels_raw.shape)\nprint(\'Test images: \', test_images_raw.shape)\nprint(\'Test labels: \', test_labels_raw.shape)\n\nassert train_images_raw.shape == (60000, 784)\nassert train_labels_raw.shape == (60000,)\nassert test_images_raw.shape == (10000, 784)\nassert test_labels_raw.shape == (10000,)\nassert set(np.unique(train_labels_raw)) == set(range(10))\nassert set(np.unique(test_labels_raw)) == set(range(10))\nprint(\'PASS: original MNIST archive shapes and label range verified.\')\n\n\n# ==========================================================================\n# 3 — Verify representative MNIST images\n# ==========================================================================\n\nfig, axes = plt.subplots(2, 5, figsize=(10, 4.5))\nfor i, ax in enumerate(axes.ravel()):\n    ax.imshow(train_images_raw[i].reshape(28, 28), cmap=\'gray\')\n    ax.set_title(f\'Label: {int(train_labels_raw[i])}\')\n    ax.axis(\'off\')\nfig.suptitle(\'First ten images in the original MNIST training data\', fontsize=13)\nfig.tight_layout()\nfig.savefig(OUT_DIR / \'mnist_examples.png\', dpi=170, bbox_inches=\'tight\')\nplt.show()\n\nprint(\'Training class counts:\', np.bincount(train_labels_raw, minlength=10))\nprint(\'Test class counts:    \', np.bincount(test_labels_raw, minlength=10))\n\n\n# ==========================================================================\n# 4 — Stratified split and input normalization\n# ==========================================================================\n\nall_indices = np.arange(train_labels_raw.shape[0])\nfit_idx, val_idx = train_test_split(\n    all_indices,\n    test_size=10000,\n    random_state=SEED,\n    stratify=train_labels_raw,\n)\n\nx_fit = train_images_raw[fit_idx].astype(np.float32) / 255.0\ny_fit = train_labels_raw[fit_idx].astype(np.int64)\nx_val = train_images_raw[val_idx].astype(np.float32) / 255.0\ny_val = train_labels_raw[val_idx].astype(np.int64)\nx_test = test_images_raw.astype(np.float32) / 255.0\ny_test = test_labels_raw.astype(np.int64)\n\nfor name, x, y in [\n    (\'Training\', x_fit, y_fit),\n    (\'Validation\', x_val, y_val),\n    (\'Test\', x_test, y_test),\n]:\n    print(f\'{name:11s}: images={x.shape}, labels={y.shape}, \'\n          f\'range=[{x.min():.3f}, {x.max():.3f}]\')\n    assert x.ndim == 2 and x.shape[1] == 784\n    assert x.dtype == np.float32\n    assert y.min() >= 0 and y.max() <= 9\n\nassert (len(x_fit), len(x_val), len(x_test)) == (50000, 10000, 10000)\nassert set(fit_idx).isdisjoint(set(val_idx))\nprint(\'PASS: train/validation/test partitions and normalization.\')\n\n\n# ==========================================================================\n# 5 — Define and verify the three Dense layers\n# ==========================================================================\n\nmodel = tf.keras.Sequential([\n    tf.keras.Input(shape=(784,), name=\'input_784\'),\n    tf.keras.layers.Dense(\n        128, activation=\'relu\', kernel_initializer=\'he_normal\', name=\'dense_128\'\n    ),\n    tf.keras.layers.Dense(\n        64, activation=\'relu\', kernel_initializer=\'he_normal\', name=\'dense_64\'\n    ),\n    tf.keras.layers.Dense(\n        10, activation=\'softmax\', name=\'class_10\'\n    ),\n], name=\'MNIST_3_Dense_Network\')\n\nmodel.summary()\nactual_dense_layers = [\n    layer for layer in model.layers if isinstance(layer, tf.keras.layers.Dense)\n]\nassert len(actual_dense_layers) == 3\nassert [layer.units for layer in actual_dense_layers] == [128, 64, 10]\nassert model.count_params() == 109386\nassert model.output_shape[-1] == 10\nassert not any(isinstance(layer, tf.keras.layers.Dropout) for layer in model.layers)\n\nprint(\'PASS: three Dense layers; 109,386 trainable parameters; no Dropout.\')\n\n\n# ==========================================================================\n# 6 — Architecture parameter table and diagram\n# ==========================================================================\n\n# Layer-structure table for the professor\'s report\nlayer_details = pd.DataFrame([\n    {\n        \'Layer\': layer.name,\n        \'Neurons\': layer.units,\n        \'Activation\': layer.activation.__name__,\n        \'Trainable parameters\': layer.count_params(),\n    }\n    for layer in actual_dense_layers\n])\nprint(layer_details.to_string(index=False))\n\nfig, ax = plt.subplots(figsize=(11, 2.7))\nax.axis(\'off\')\narchitecture = [\n    (\'Input\', \'784 pixels\'),\n    (\'Dense 1\', \'128 · ReLU\'),\n    (\'Dense 2\', \'64 · ReLU\'),\n    (\'Dense 3\', \'10 · Softmax\'),\n]\nfor i, (title, detail) in enumerate(architecture):\n    x = 0.115 + i * 0.25\n    ax.text(x, 0.6, title, ha=\'center\', fontsize=12, weight=\'bold\',\n            bbox=dict(boxstyle=\'round,pad=.7\', fc=\'#eef3f9\', ec=\'#38577a\'))\n    ax.text(x, 0.26, detail, ha=\'center\', fontsize=11)\n    if i < 3:\n        ax.annotate(\'\', xy=(x+0.21, .59), xytext=(x+0.10, .59),\n                    arrowprops=dict(arrowstyle=\'->\', lw=1.5))\nfig.suptitle(\'Three fully connected layers (input not counted as a Dense layer)\',\n             fontsize=13)\nfig.tight_layout()\nfig.savefig(OUT_DIR / \'architecture.png\', dpi=180, bbox_inches=\'tight\')\nplt.show()\n\n\n# ==========================================================================\n# 7 — Compile with loss, optimizer and hyperparameters\n# ==========================================================================\n\nINITIAL_LEARNING_RATE = 0.001\nBATCH_SIZE = 128\nMAX_EPOCHS = 15\nEARLY_STOP_PATIENCE = 3\nDROPOUT_USED = False\n\noptimizer = tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE)\nloss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)\n\nmodel.compile(\n    optimizer=optimizer,\n    loss=loss_fn,\n    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name=\'accuracy\')],\n)\n\ncallbacks = [\n    tf.keras.callbacks.EarlyStopping(\n        monitor=\'val_loss\', patience=EARLY_STOP_PATIENCE,\n        restore_best_weights=True, verbose=1,\n    ),\n    tf.keras.callbacks.TerminateOnNaN(),\n]\n\nprint(\'Optimizer: Adam\')\nprint(\'Initial learning rate:\', INITIAL_LEARNING_RATE)\nprint(\'Loss:\', type(loss_fn).__name__)\nprint(\'Epoch limit:\', MAX_EPOCHS, \'; batch size:\', BATCH_SIZE)\nprint(\'EarlyStopping patience:\', EARLY_STOP_PATIENCE)\nprint(\'Dropout used:\', DROPOUT_USED)\n\n\n# ==========================================================================\n# 8 — Train on training partition with validation and early stopping\n# ==========================================================================\n\nhistory = model.fit(\n    x_fit, y_fit,\n    validation_data=(x_val, y_val),\n    epochs=MAX_EPOCHS,\n    batch_size=BATCH_SIZE,\n    callbacks=callbacks,\n    shuffle=True,\n    verbose=2,\n)\n\nobserved_epochs = len(history.history[\'loss\'])\nbest_epoch = int(np.argmin(history.history[\'val_loss\']) + 1)\nprint(\'Epochs actually run:\', observed_epochs)\nprint(\'Best validation-loss epoch:\', best_epoch)\n\n\n# ==========================================================================\n# 9 — Plot and interpret training history\n# ==========================================================================\n\nfig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\nepochs = np.arange(1, observed_epochs + 1)\n\naxes[0].plot(epochs, history.history[\'loss\'], marker=\'o\', label=\'Training\')\naxes[0].plot(epochs, history.history[\'val_loss\'], marker=\'o\', label=\'Validation\')\naxes[0].set_title(\'Sparse categorical cross-entropy\')\naxes[0].set_xlabel(\'Epoch\'); axes[0].set_ylabel(\'Loss\')\naxes[0].legend(); axes[0].grid(alpha=0.3)\n\naxes[1].plot(epochs, history.history[\'accuracy\'], marker=\'o\', label=\'Training\')\naxes[1].plot(epochs, history.history[\'val_accuracy\'], marker=\'o\', label=\'Validation\')\naxes[1].set_title(\'Classification accuracy\')\naxes[1].set_xlabel(\'Epoch\'); axes[1].set_ylabel(\'Accuracy (fraction)\')\naxes[1].legend(); axes[1].grid(alpha=0.3)\n\nfig.suptitle(\'MNIST three-Dense-layer training history\')\nfig.tight_layout()\nfig.savefig(OUT_DIR / \'learning_curves.png\', dpi=180, bbox_inches=\'tight\')\nplt.show()\n\nprint(\'Final logged training accuracy: \', f"{history.history[\'accuracy\'][-1]:.2%}")\nprint(\'Final logged validation accuracy:\', f"{history.history[\'val_accuracy\'][-1]:.2%}")\nprint(\'Best validation loss:\', f"{min(history.history[\'val_loss\']):.6f}")\n\n\n# ==========================================================================\n# 10 — Independent test evaluation and confusion matrix\n# ==========================================================================\n\n# Evaluate once after training/model selection.\ntest_loss, test_accuracy_fraction = model.evaluate(\n    x_test, y_test, batch_size=512, verbose=0\n)\n\nprobabilities = model.predict(x_test, batch_size=512, verbose=0)\ny_pred = np.argmax(probabilities, axis=1).astype(np.int64)\ncm = confusion_matrix(y_test, y_pred, labels=np.arange(10))\n\nnumber_correct = int(np.count_nonzero(y_pred == y_test))\ntest_accuracy_percent = 100.0 * number_correct / len(y_test)\n\nassert cm.shape == (10, 10)\nassert int(cm.sum()) == len(y_test) == 10000\nassert np.allclose(probabilities.sum(axis=1), 1, atol=1e-4)\nassert abs(test_accuracy_percent/100 - test_accuracy_fraction) < 2e-3\n\nprint(\'OFFICIAL MNIST TEST RESULTS\')\nprint(\'Test cross-entropy:\', f\'{test_loss:.6f}\')\nprint(\'Correct predictions:\', f\'{number_correct:,} / 10,000\')\nprint(\'TEST ACCURACY:\', f\'{test_accuracy_percent:.2f}%\')\nprint(\'\\nConfusion matrix (rows = actual; columns = predicted):\')\nprint(pd.DataFrame(cm, index=np.arange(10), columns=np.arange(10)).to_string())\n\n\n# ==========================================================================\n# 11 — Plot the confusion matrix\n# ==========================================================================\n\nfig, ax = plt.subplots(figsize=(9, 7.5))\nim = ax.imshow(cm, interpolation=\'nearest\', cmap=\'Blues\')\nfig.colorbar(im, ax=ax, fraction=.045)\nax.set(\n    xticks=np.arange(10), yticks=np.arange(10),\n    xlabel=\'Predicted digit\', ylabel=\'True digit\',\n    title=\'MNIST 10 × 10 test confusion matrix\',\n)\nthreshold = cm.max() / 2\nfor row in range(10):\n    for col in range(10):\n        ax.text(col, row, str(cm[row, col]), ha=\'center\', va=\'center\',\n                fontsize=9, color=\'white\' if cm[row, col] > threshold else \'black\')\nfig.tight_layout()\nfig.savefig(OUT_DIR / \'confusion_matrix.png\', dpi=200, bbox_inches=\'tight\')\nplt.show()\n\n\n# ==========================================================================\n# 12 — Per-class performance and metrics exports\n# ==========================================================================\n\nreport_dictionary = classification_report(\n    y_test, y_pred, labels=list(range(10)),\n    output_dict=True, zero_division=0,\n)\nper_digit = pd.DataFrame([\n    {\n        \'Digit\': digit,\n        \'Test count\': int(cm[digit].sum()),\n        \'Correct\': int(cm[digit, digit]),\n        \'Recall\': float(report_dictionary[str(digit)][\'recall\']),\n        \'Precision\': float(report_dictionary[str(digit)][\'precision\']),\n        \'F1\': float(report_dictionary[str(digit)][\'f1-score\']),\n    }\n    for digit in range(10)\n])\nprint(per_digit.to_string(index=False, float_format=lambda v: f\'{v:.4f}\'))\nper_digit.to_csv(OUT_DIR / \'per_digit_metrics.csv\', index=False)\nnp.savetxt(OUT_DIR / \'confusion_matrix.csv\', cm, fmt=\'%d\', delimiter=\',\')\n\n\n# ==========================================================================\n# 13 — Inspect representative classification errors\n# ==========================================================================\n\nwrong = np.flatnonzero(y_pred != y_test)\nfig, axes = plt.subplots(2, 5, figsize=(10.5, 5))\nfor ax, idx in zip(axes.ravel(), wrong[:10]):\n    ax.imshow(x_test[idx].reshape(28,28), cmap=\'gray\')\n    ax.set_title(f\'True {y_test[idx]}  /  Pred {y_pred[idx]}\', fontsize=10)\n    ax.axis(\'off\')\nfor ax in axes.ravel()[len(wrong[:10]):]:\n    ax.axis(\'off\')\nfig.suptitle(\'Examples of incorrect test predictions (when present)\')\nfig.tight_layout()\nfig.savefig(OUT_DIR / \'mistakes.png\', dpi=170, bbox_inches=\'tight\')\nplt.show()\nprint(\'Number of test mistakes:\', len(wrong))\n\n\n# ==========================================================================\n# 14 — Evidence-based underfitting/overfitting interpretation\n# ==========================================================================\n\n# Dynamic, evidence-based comments; do not hard-code a model quality claim.\nlast_fit_acc = float(history.history[\'accuracy\'][-1])\nlast_val_acc = float(history.history[\'val_accuracy\'][-1])\ngap = last_fit_acc - last_val_acc\n\nprint(\'TRAINING DIAGNOSTICS\')\nprint(f\'Final logged training accuracy: {last_fit_acc:.2%}\')\nprint(f\'Final logged validation accuracy: {last_val_acc:.2%}\')\nprint(f\'Train minus validation accuracy: {gap:+.2%}\')\nprint(f\'Best validation-loss epoch: {best_epoch} of {observed_epochs} run\')\nif gap > 0.03:\n    print(\'Interpretation: a >3 percentage-point gap may indicate overfitting.\')\nelif last_fit_acc < 0.85 and last_val_acc < 0.85:\n    print(\'Interpretation: both accuracies are low; inspect underfitting or training setup.\')\nelse:\n    print(\'Interpretation: no strong accuracy-gap warning by this simple heuristic.\')\nprint(\'This heuristic is not a formal statistical test of generalization.\')\n\n\n# ==========================================================================\n# 15 — Assemble measured answers to all four report requirements\n# ==========================================================================\n\nreport_summary = {\n    \'test_accuracy_percent\': float(test_accuracy_percent),\n    \'test_loss\': float(test_loss),\n    \'correct\': int(number_correct),\n    \'test_total\': int(len(y_test)),\n    \'confusion_matrix\': cm.copy(),\n    \'network\': \'784 inputs -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(10, Softmax)\',\n    \'parameter_count\': int(model.count_params()),\n    \'loss_name\': \'Sparse categorical cross-entropy\',\n    \'optimizer\': \'Adam\',\n    \'initial_learning_rate\': INITIAL_LEARNING_RATE,\n    \'batch_size\': BATCH_SIZE,\n    \'epochs_run\': observed_epochs,\n    \'epoch_limit\': MAX_EPOCHS,\n    \'best_validation_epoch\': best_epoch,\n    \'validation_patience\': EARLY_STOP_PATIENCE,\n    \'dropout_used\': False,\n    \'final_training_accuracy\': last_fit_acc,\n    \'final_validation_accuracy\': last_val_acc,\n}\n\nprint(\'REQUIREMENT 1 — TEST ACCURACY AND CONFUSION MATRIX\')\nprint(f"Accuracy: {report_summary[\'test_accuracy_percent\']:.2f}% "\n      f"({report_summary[\'correct\']:,}/{report_summary[\'test_total\']:,})")\nprint(pd.DataFrame(cm).to_string(index=False, header=False))\n\nprint(\'\\nREQUIREMENT 2 — NETWORK STRUCTURE\')\nprint(report_summary[\'network\'])\nprint(\'Trainable parameters:\', report_summary[\'parameter_count\'])\n\nprint(\'\\nREQUIREMENT 3 — LOSS\')\nprint(\'Sparse categorical cross-entropy (integer labels 0–9)\')\n\nprint(\'\\nREQUIREMENT 4 — TRAINING / DROPOUT\')\nprint(\'Adam, initial lr=\', INITIAL_LEARNING_RATE, \', batch=\', BATCH_SIZE,\n      \', maximum epochs=\', MAX_EPOCHS, \', observed epochs=\', observed_epochs)\nprint(\'Early stopping: validation loss, patience 3; best epoch=\', best_epoch)\nprint(\'Dropout: Not used; professor says optional.\')\n\n\n# ==========================================================================\n# 16 — Final run-time audit\n# ==========================================================================\n\nassert train_images_raw.shape == (60000, 784)\nassert test_images_raw.shape == (10000, 784)\nassert len(actual_dense_layers) == 3\nassert [layer.units for layer in actual_dense_layers] == [128, 64, 10]\nassert model.count_params() == 109386\nassert np.all(np.isfinite(probabilities))\nassert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4)\nassert cm.shape == (10, 10) and cm.sum() == 10000\nassert np.array_equal(cm.sum(axis=1), np.bincount(y_test, minlength=10))\nassert cm.trace() == number_correct\nassert abs(cm.trace()/cm.sum() - test_accuracy_percent/100) < 1e-12\nassert not DROPOUT_USED\n\nprint(\'ALL AVAILABLE MATHEMATICAL AND INSTRUCTION-COMPLIANCE CHECKS PASSED.\')\nprint(\'These checks verify internal consistency, not a guaranteed professor grade.\')\n'

source_py = OUT_DIR / 'MENG417_HW4_Submission_Source.py'
source_py.write_text(EXPERIMENT_SOURCE, encoding='utf-8')
report_pdf = OUT_DIR / 'MENG417_HW4_FINAL_Submission.pdf'

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name='RptTitle', parent=styles['Title'],
    fontSize=18, leading=22, alignment=TA_CENTER, spaceAfter=10))
styles.add(ParagraphStyle(name='RptH', parent=styles['Heading2'],
    fontSize=12, leading=16, spaceBefore=10, spaceAfter=6,
    textColor=colors.HexColor('#18375A')))
styles.add(ParagraphStyle(name='RptBody', parent=styles['BodyText'],
    fontSize=9.1, leading=13, spaceAfter=6))
styles.add(ParagraphStyle(name='RptCode', fontName='Courier', fontSize=6.5,
    leading=8.7, leftIndent=4, spaceAfter=2))

pdf = SimpleDocTemplate(str(report_pdf), pagesize=A4,
    leftMargin=17*mm, rightMargin=17*mm,
    topMargin=15*mm, bottomMargin=15*mm,
    title='MENG 417 Homework 4: Complete MNIST Neural Network Submission')

items = []
def p(s, style='RptBody'):
    items.append(Paragraph(s, styles[style]))
def figpdf(name, width=165*mm):
    path = OUT_DIR / name
    if path.exists():
        from PIL import Image as PILImage
        with PILImage.open(path) as im:
            w, h = im.size
        items.append(Image(str(path), width=width, height=width*h/w))
        items.append(Spacer(1, 4*mm))

def rule_table(data, col_widths):
    t = Table(data, colWidths=col_widths, repeatRows=1)
    t.setStyle(TableStyle([
        ('BACKGROUND', (0,0),(-1,0), colors.HexColor('#E7EDF5')),
        ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
        ('GRID',(0,0),(-1,-1),0.3,colors.HexColor('#B9C5D0')),
        ('VALIGN',(0,0),(-1,-1),'TOP'),
        ('FONTSIZE',(0,0),(-1,-1),8),
        ('TOPPADDING',(0,0),(-1,-1),4),
        ('BOTTOMPADDING',(0,0),(-1,-1),4),
    ]))
    items.append(t)
    items.append(Spacer(1,5*mm))

p('MENG 417 — Homework #4', 'RptTitle')
p('Three Fully Connected Layers for MNIST Digit Classification')
p('Experiment: original MNIST IDX archives; 60,000 official training examples, '
  '10,000 official test examples. Internal stratified split: 50,000 fitting examples '
  'and 10,000 validation examples. The final test set was not used for model selection.')

p('1. Test accuracy and confusion matrix', 'RptH')
p(f'The trained classifier correctly classified <b>{number_correct:,} of 10,000</b> '
  f'official test images. Test accuracy: <b>{test_accuracy_percent:.2f}%</b>. '
  f'Test sparse cross-entropy: <b>{test_loss:.6f}</b>.')
p(f'Accuracy check: {number_correct:,} / 10,000 = {test_accuracy_percent:.2f}%.')
p('Rows of the confusion matrix are the true digits 0–9; columns are the '
  'predicted digits 0–9. The total of all cells is 10,000.')
figpdf('confusion_matrix.png', width=137*mm)
p('2. Three-layer network structure', 'RptH')
p('The input has 784 grayscale pixel values (28 by 28). '
  'The three trainable fully connected layers are 128 ReLU neurons, '
  '64 ReLU neurons and 10 Softmax neurons. No convolutional, pooling, '
  'normalization or Dropout layers were used.')
rule_table([
    ['Layer', 'Shape', 'Activation', 'Parameters'],
    ['Dense 1', '784 to 128', 'ReLU', '100,480'],
    ['Dense 2', '128 to 64', 'ReLU', '8,256'],
    ['Dense 3', '64 to 10', 'Softmax', '650'],
    ['Total', '', '', '109,386'],
], [35*mm, 44*mm, 42*mm, 40*mm])
figpdf('architecture.png', width=160*mm)
p('3. Loss function and model learning', 'RptH')
p('We used <b>sparse categorical cross-entropy</b>, appropriate for ten mutually '
  'exclusive classes labeled with integers 0–9. The per-example loss equals '
  'minus the natural logarithm of the predicted probability assigned to the '
  'correct class; the mini-batch loss is the average over examples. '
  'In forward propagation, the two hidden layers use ReLU activations and '
  'the output layer uses Softmax probabilities. TensorFlow backpropagates '
  'cross-entropy derivatives through these layers to obtain weight gradients.')
p('4. Training hyperparameters, optimizer, initialization, Dropout', 'RptH')
rule_table([
    ['Setting', 'Used in the experiment'],
    ['Optimizer', 'Adam'],
    ['Initial learning rate', f'{INITIAL_LEARNING_RATE:g}'],
    ['Batch size', str(BATCH_SIZE)],
    ['Maximum epochs', str(MAX_EPOCHS)],
    ['Epochs actually run', str(observed_epochs)],
    ['Best validation-loss epoch', str(best_epoch)],
    ['Early stopping', f'Validation loss; patience {EARLY_STOP_PATIENCE}; restore best weights'],
    ['Input scaling', 'uint8 divided by 255; float32'],
    ['Hidden-layer initialization', 'He normal'],
    ['Dropout', 'Not used (optional according to professor)'],
], [61*mm, 102*mm])
p('Early stopping on validation loss was used to reduce unnecessary '
  'training. Dropout was <b>not</b> included, as the homework does not '
  'require it. Dropout would randomly suppress some hidden activations '
  'during training to discourage co-adaptation; it is normally disabled '
  'at inference time. Pixel scaling, ReLU-compatible He initialization, '
  'numerically implemented Softmax and cross-entropy, and Adam epsilon '
  'help improve numerical stability.')
p('Training and generalization assessment', 'RptH')
p(f'Final logged training accuracy: {last_fit_acc:.2%}; final logged '
  f'validation accuracy: {last_val_acc:.2%}; difference: {gap:+.2%}. '
  'The learning curves below provide evidence for interpreting '
  'underfitting or overfitting; a simple accuracy gap alone is not '
  'conclusive proof of either condition.')
figpdf('learning_curves.png', width=162*mm)
figpdf('mistakes.png', width=145*mm)

items.append(CondPageBreak(80*mm))
p('Appendix — Complete experiment Python source code', 'RptH')
p('The following source contains the code used to load the four raw MNIST '
  'files, prepare the data, build the three-layer model, train it, '
  'and calculate the evaluation figures in this report.')

# Preserve indentation; break into moderate blocks so the appendix paginates.
lines = EXPERIMENT_SOURCE.replace('\t', '    ').splitlines()
for start in range(0, len(lines), 42):
    block = '\n'.join(lines[start:start+42])
    # Long lines are wrapped for print legibility without changing source file.
    import textwrap
    pretty_lines = []
    for line in block.splitlines():
        if len(line) > 110:
            pretty_lines += textwrap.wrap(line, 110, subsequent_indent='    ',
                replace_whitespace=False, drop_whitespace=False)
        else:
            pretty_lines.append(line)
    items.append(Preformatted('\n'.join(pretty_lines), styles['RptCode'],
                              maxLineLength=112, splitChars=' '))

def add_footer(canvas, doc):
    canvas.saveState()
    canvas.setFont('Helvetica', 8)
    canvas.setFillColor(colors.HexColor('#667282'))
    canvas.drawRightString(A4[0] - 17*mm, 8*mm, f'Page {doc.page}')
    canvas.restoreState()

pdf.build(items, onFirstPage=add_footer, onLaterPages=add_footer)
print('FINAL PDF:', report_pdf, f'({report_pdf.stat().st_size:,} bytes)')
print('PYTHON SOURCE:', source_py)
print('Submit the one combined PDF after reviewing its measured contents.')

# Optional Google Colab download; Library/Drive also works from the Files sidebar.
try:
    from google.colab import files
    files.download(str(report_pdf))
except ImportError:
    print('Download the PDF from the notebook working directory:', report_pdf)

# Complete theory cheat sheet — MENG 417 Homework #4

## 1. What is a Dense neural-network layer?

A Dense layer connects **every incoming feature** to **every neuron** in the next layer. It calculates

$$
\boxed{\mathbf z=W^T\mathbf x+\mathbf b.}
$$

An activation function then produces the layer output.

## 2. Why use three fully connected layers?

- First Dense layer (128 neurons): learns patterns from 784 pixels.
- Second Dense layer (64 neurons): combines those patterns.
- Third Dense layer (10 neurons): gives probabilities for the ten possible digits.

$$
\boxed{784\rightarrow128\rightarrow64\rightarrow10}
$$

## 3. Why ReLU?

$$
\boxed{\mathrm{ReLU}(z)=\max(0,z).}
$$

It introduces nonlinearity and is simple to differentiate almost everywhere.

## 4. Why Softmax?

$$
\boxed{p_k=\frac{e^{z_k}}{\sum_{j=0}^{9}e^{z_j}}.}
$$

Softmax gives ten nonnegative scores summing to 1, allowing multiclass classification.

## 5. Why cross-entropy instead of MSE?

$$
\boxed{\ell=-\log p_{\mathrm{true\ digit}}.}
$$

It penalizes assigning low probability to the correct class and naturally matches categorical labels. Sparse cross-entropy accepts integer targets, so we do not need one-hot labels.

## 6. What is backpropagation?

It is the systematic application of the chain rule through the computational graph **backwards** to calculate gradients. The optimizer uses those gradients to update the network weights.

## 7. Parameters vs hyperparameters

**Parameters** are trained values: the **109,386 weights and biases**.

**Hyperparameters** are chosen settings: hidden-unit counts, optimizer, learning rate, batch size, epoch limit and early-stopping patience.

## 8. What is the difference between training, validation and testing?

Training updates weights; validation influences training decisions (such as stopping epoch); testing gives an independent final estimate of classification performance.

## 9. How do underfitting and overfitting differ?

- **Underfitting:** the model does not learn sufficiently even on its training data.
- **Overfitting:** training results improve but generalization to held-out data gets worse.
- **Model selection:** use validation data to choose settings, not the independent test data.

## 10. What does Dropout do?

It randomly zeros some activations **during training** and can reduce overfitting. It is **not required** for Homework #4, so our model did not use it.

## 11. Why is initialization important?

If weights are badly scaled, information or gradients can explode or vanish. He initialization is suitable for ReLU networks and helps stabilize early training.

## 12. Why is numerical stability important?

Scaling inputs, using stable framework loss implementations, and adding optimizer epsilon help avoid overflow, underflow or unstable learning.

## 13. What does a confusion matrix show?

It counts the correct and incorrect predictions for each pair of true and predicted digits.

$$
\boxed{\text{rows = true digits; columns = predicted digits}.}
$$

Diagonal = correct; off-diagonal = errors; sum of all cells = 10,000.

## 14. Last pre-submission checklist

- [ ] Four original MNIST `.gz` files loaded via the IDX decoder; no built-in Keras MNIST loader.
- [ ] Exactly three Dense layers with 10 output neurons.
- [ ] All code cells executed without errors on original MNIST.
- [ ] Real test accuracy printed from 10,000 test cases.
- [ ] Confusion-matrix plot contains ten labeled rows and columns.
- [ ] Loss, optimizer, initial learning rate, epochs, batch size and Dropout documented.
- [ ] The final **one PDF** regenerated and inspected; appendix includes the source code.

**Important:** Passing the checks confirms internal consistency, not a guarantee of the professor's score. Never replace measured results with made-up percentages.